In [7]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [8]:
URL_BASE = "https://rorystravelclub.com/pages/rtc-"
 
HEADERS = {
    # A normal browser UA avoids some basic bot-blocking
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

CARD_CLASS_RE = re.compile(r"-offers-card-[a-zA-Z0-9]+$")


COUNTY_TO_PROVINCE = {
    "carlow": "Leinster", "dublin": "Leinster", "kildare": "Leinster", "kilkenny": "Leinster",
    "laois": "Leinster", "longford": "Leinster", "louth": "Leinster", "meath": "Leinster",
    "offaly": "Leinster", "westmeath": "Leinster", "wexford": "Leinster", "wicklow": "Leinster",
    "clare": "Munster", "cork": "Munster", "kerry": "Munster", "limerick": "Munster",
    "tipperary": "Munster", "waterford": "Munster",
    "galway": "Connacht", "leitrim": "Connacht", "mayo": "Connacht", "roscommon": "Connacht",
    "sligo": "Connacht",
    "cavan": "Ulster", "donegal": "Ulster", "monaghan": "Ulster",
}

In [9]:
def fetch_html(url: str) -> str:
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text

In [10]:
def parse_prices(text: str):
    """Extract all €/£ amounts from a deal description and return the min."""
    amounts = re.findall(r"[€£]\s?(\d[\d,]*(?:\.\d{1,2})?)", text)
    if not amounts:
        return None, None
    currency = "€" if "€" in text else ("£" if "£" in text else None)
    values = [float(a.replace(",", "")) for a in amounts]
    return currency, min(values)    

In [11]:
def parse_expiry(text: str):
    """'Valid until: August 31, 2026' -> Timestamp, or None if unparsable."""
    match = re.search(r"Valid until:\s*(.+)", text, re.IGNORECASE)
    if not match:
        return None
    try:
        return pd.to_datetime(match.group(1).strip(), errors="coerce")
    except Exception:
        return None

In [14]:
records = []

locations = ['leinster-offers', 'munster-offers', 'connacht-offers', 'ulster-offers-1']

for location in locations:
    url = URL_BASE + location
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    # NOTE: bs4 calls class_ functions once per individual class token, not
    # once with the full class list -- so `c` here is a single string.
    cards = soup.find_all(
        "div",
        class_=lambda c: c and CARD_CLASS_RE.search(c),
    )

    for card in cards:
        # print(card.prettify())

        # Hotel name
        hotel = card.get("data-hotel") or card.find("h3").get_text(strip=True)

        # County
        county = card.get("data-county")

        # Image
        img = card.find("img")
        image_url = img.get("src") if img else None
        if image_url and image_url.startswith("//"):
            image_url = "https:" + image_url

        # Description (preserve line breaks)
        desc = card.find("p", class_=lambda c: c and "description" in c)
        description = (
            desc.get_text(separator="\n", strip=True)
            if desc else None
        )

        # Expiry
        expiry = card.find("p", class_=lambda c: c and "expiry" in c)
        expiry = (
            expiry.get_text(strip=True).replace("Valid until:", "").strip()
            if expiry else None
        )

        # Offer URL
        link = card.find("a", href=True)
        offer_url = link["href"] if link else None
        
        # Province mapping
        province = province = COUNTY_TO_PROVINCE.get(county.lower(), "Unknown")

        records.append({
            "hotel": hotel,
            "county": county,
            "province": province,
            "description": description,
            "expiry": expiry,
            "offer_url": offer_url,
            "image_url": image_url,
        })

df = pd.DataFrame(records)

df.to_csv("rorystravelclub_offers.csv", index=False)

In [13]:
df.head()

,hotel,county,province,description,expiry,offer_url,image_url
0,osprey hotel & spa retreat,kildare,Leinster,"Rory’s Summer Retreat: B&B, Welcome Drink, Mai...","August 31, 2026",https://rorystravelclub.com/pages/ospreyhotela...,https://rorystravelclub.com/cdn/shop/files/Osp...
1,osprey hotel & spa retreat,kildare,Leinster,"Rory's Stay, Dine and Pamper\nB&B, Welcome Dri...","August 31, 2026",https://rorystravelclub.com/pages/ospreyhotela...,https://rorystravelclub.com/cdn/shop/files/Osp...
2,glashaus hotel,dublin,Leinster,15% OFF Room Only and B&B Rates,"May 05, 2027",https://rorystravelclub.com/pages/glashaushotel,https://rorystravelclub.com/cdn/shop/files/gla...
3,plaza hotel tallaght,dublin,Leinster,"1 Night Package, 2 Night Package & Family Geta...","September 30, 2026",https://rorystravelclub.com/pages/landingpagep...,https://rorystravelclub.com/cdn/shop/files/606...
4,ashdown park hotel,wexford,Leinster,"B&B, 2 Course Meal, Access to Leisure Faciliti...","August 31, 2026",https://rorystravelclub.com/pages/ashdownparkh...,https://rorystravelclub.com/cdn/shop/files/hot...


In [15]:
!pip install streamlit

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 2.9 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 2.9 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 2.8 MB/s  0:00:04 eta 0:00:01
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached mar